In [61]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from plotnine.ggplot import ggplot
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import mofaflex as mfl
import itertools

# pipeline to evaluate a MOFAFLEX model and lrdata.

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, silhouette_score,
    normalized_mutual_info_score, adjusted_mutual_info_score,
    homogeneity_score, adjusted_rand_score
)


def _ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def spatial_coherence_score(X, labels, k=5):
    X = np.asarray(X)
    labels = np.asarray(labels)
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, indices = nbrs.kneighbors(X)
    indices = indices[:, 1:]
    same_cluster_counts = np.sum(labels[indices] == labels[:, None], axis=1)
    scs = np.mean(same_cluster_counts / k)
    return scs


def save_plot_object(fig, path):
    # supports plotnine and matplotlib
    if isinstance(fig, ggplot):
        fig.save(path, dpi=300)
    else:
        fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def evaluate_single_mofa_model(model, output_dir, group="group_1", show_plots=False):
    out = {}
    out_dir = _ensure_dir(output_dir)

    # Factor correlation
    try:
        fig = mfl.pl.factor_correlation(model)
        p = out_dir / "factor_correlation.png"
        save_plot_object(fig, p)
        out["factor_correlation_path"] = str(p)
    except Exception as e:
        out["factor_correlation_error"] = str(e)

    # Variance explained
    try:
        fig = mfl.pl.variance_explained(model, figsize=(8, 8))
        p = out_dir / "variance_explained.png"
        save_plot_object(fig, p)
        out["variance_explained_path"] = str(p)
    except Exception as e:
        out["variance_explained_error"] = str(e)

    # Mean R2
    try:
        r2 = model.get_r2(total=True)
        mean_r2 = float(r2[group].mean())
        out["mean_r2"] = mean_r2
    except Exception as e:
        out["mean_r2_error"] = str(e)

    return out


def plot_weights_distribution(model, output_dir):
    out_dir = _ensure_dir(output_dir)
    try:
        weights_dict = model.get_weights()
        plt.figure(figsize=(12, 6))
        for key, df in weights_dict.items():
            values = df.values.flatten()
            sns.kdeplot(values, label=key, fill=False)
        plt.title("Distribution of Weights per View")
        plt.xlabel("Weight Value")
        plt.ylabel("Density")
        plt.legend(title="View/Key", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        p = out_dir / "weights_distribution.png"
        plt.savefig(p, bbox_inches="tight", dpi=150)
        plt.close()
        return str(p)
    except Exception as e:
        return {"error": str(e)}


def clustering_sweep(factors, lrdata_sub, output_dir, n_clusters_list=None, spatial_k_for_scs=3,
                     obs_keys=None):
    """
    Run clustering sweep and compute metrics comparing clusters to any number of observation keys.

    Parameters
    ----------
    obs_keys : list of str or None
        Observation keys in lrdata_sub.obs to compare clusters against. If None, no obs-based metrics are computed.
    """
    out_dir = _ensure_dir(output_dir)
    if n_clusters_list is None:
        n_clusters_list = list(range(2, 10))
    if obs_keys is None:
        obs_keys = []

    # Prepare X_f (drop non-factor cols if present)
    drop_cols = [c for c in ["cluster", "anno"] if c in factors.columns]
    X_f = factors.drop(columns=drop_cols)
    X_std = StandardScaler().fit_transform(X_f.values)

    # spatial coords
    X_spatial = lrdata_sub.obsm["X_spatial_coords"]
    # align ordering
    common_idx = factors.index.intersection(lrdata_sub.obs_names)
    if not list(common_idx) == list(lrdata_sub.obs_names):
        X_spatial = lrdata_sub.obsm["X_spatial_coords"][[lrdata_sub.obs_names.get_loc(i) for i in common_idx], :]

    results = []
    # pre-extract obs arrays for keys to avoid repeated conversion
    obs_arrays = {}
    for key in obs_keys:
        if key in lrdata_sub.obs:
            obs_arrays[key] = lrdata_sub.obs[key].astype(str).values
        else:
            obs_arrays[key] = None

    # prepare pairwise key combinations for cross-key metrics
    pairwise_keys = list(itertools.combinations(obs_keys, 2))

    for k in n_clusters_list:
        km = KMeans(n_clusters=k, random_state=42)
        labels = km.fit_predict(X_std)

        row = {"n_clusters": k}

        # silhouette scores
        try:
            row["silhouette_spatial"] = float(silhouette_score(X_spatial, labels))
        except Exception:
            row["silhouette_spatial"] = float("nan")

        # metrics per obs_key
        for key, arr in obs_arrays.items():
            pref = key.replace(" ", "_")
            if arr is None:
                row[f"silhouette_{pref}"] = float("nan")
                row[f"nmi_cluster_{pref}"] = float("nan")
                row[f"ami_cluster_{pref}"] = float("nan")
                row[f"hom_cluster_{pref}"] = float("nan")
                row[f"ari_cluster_{pref}"] = float("nan")
                row[f"spatial_coherence_score_{pref}"] = float("nan")
                continue

            try:
                row[f"silhouette_{pref}"] = float(silhouette_score(X_spatial, arr))
            except Exception:
                row[f"silhouette_{pref}"] = float("nan")

            try:
                row[f"nmi_cluster_{pref}"] = float(normalized_mutual_info_score(arr, labels))
            except Exception:
                row[f"nmi_cluster_{pref}"] = float("nan")
            try:
                row[f"ami_cluster_{pref}"] = float(adjusted_mutual_info_score(arr, labels))
            except Exception:
                row[f"ami_cluster_{pref}"] = float("nan")
            try:
                row[f"hom_cluster_{pref}"] = float(homogeneity_score(arr, labels))
            except Exception:
                row[f"hom_cluster_{pref}"] = float("nan")
            try:
                row[f"ari_cluster_{pref}"] = float(adjusted_rand_score(arr, labels))
            except Exception:
                row[f"ari_cluster_{pref}"] = float("nan")
            try:
                row[f"spatial_coherence_score_{pref}"] = float(spatial_coherence_score(X_spatial, arr, k=spatial_k_for_scs))
            except Exception:
                row[f"spatial_coherence_score_{pref}"] = float("nan")

        # pairwise metrics between obs_keys
        for a, b in pairwise_keys:
            arr_a = obs_arrays.get(a)
            arr_b = obs_arrays.get(b)
            pref = f"{a.replace(' ','_')}_vs_{b.replace(' ','_')}"
            if arr_a is None or arr_b is None:
                row[f"nmi_{pref}"] = float("nan")
                row[f"ami_{pref}"] = float("nan")
                row[f"hom_{pref}"] = float("nan")
                row[f"ari_{pref}"] = float("nan")
                continue
            try:
                row[f"nmi_{pref}"] = float(normalized_mutual_info_score(arr_a, arr_b))
            except Exception:
                row[f"nmi_{pref}"] = float("nan")
            try:
                row[f"ami_{pref}"] = float(adjusted_mutual_info_score(arr_a, arr_b))
            except Exception:
                row[f"ami_{pref}"] = float("nan")
            try:
                row[f"hom_{pref}"] = float(homogeneity_score(arr_a, arr_b))
            except Exception:
                row[f"hom_{pref}"] = float("nan")
            try:
                row[f"ari_{pref}"] = float(adjusted_rand_score(arr_a, arr_b))
            except Exception:
                row[f"ari_{pref}"] = float("nan")

        results.append(row)

    results_df = pd.DataFrame(results).set_index("n_clusters").sort_index()
    csvp = out_dir / "clustering_metrics_sweep.csv"
    results_df.to_csv(csvp)
    return results_df, str(csvp)


def classification_on_factors(factors, lrdata, output_dir, label_key=None, test_size=0.3, random_state=42):
    out_dir = _ensure_dir(output_dir)
    # prepare labels
    if label_key not in lrdata.obs:
        raise KeyError(f"label_key '{label_key}' not found in lrdata.obs")
    obskey1 = lrdata.obs[label_key]
    factors = factors.copy()
    factors["anno"] = obskey1.reindex(factors.index)
    factors = factors.dropna()
    # drop potential non-factor cols
    drop_cols = [c for c in ["cluster", "anno"] if c in factors.columns]
    X = factors.drop(columns=drop_cols + ([] if "anno" not in factors.columns else [])).values
    y = factors["anno"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    clf = LogisticRegression(solver="lbfgs", max_iter=200, class_weight="balanced")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cls_report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
    cm_df.to_csv(out_dir / "confusion_matrix.csv")

    # ROC AUC (multiclass supported)
    try:
        y_proba = clf.predict_proba(X_test)
        auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
        auc_weighted = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
    except Exception:
        auc_macro = np.nan
        auc_weighted = np.nan

    # save metrics
    metrics = {
        "accuracy": acc,
        "auc_macro": auc_macro,
        "auc_weighted": auc_weighted,
        "classification_report": cls_report,
    }
    pd.DataFrame(cls_report).to_csv(out_dir / "classification_report.csv")
    return metrics, str(out_dir / "classification_report.csv")


def evaluate_pipeline(model, lrdata, output_dir, group="group_1",
                      n_clusters_list=None, obs_keys=None, label_key=None):
    """
    Evaluates a MOFAFLEX model and associated lrdata, generating plots, metrics, classification metrics and report, and summary statistics.

    Parameters
    ----------
    model : MOFAFLEX model object
        The trained MOFAFLEX model to evaluate.
    lrdata : AnnData-like object
        The data object containing observations and spatial coordinates.
    output_dir : str or Path
        Directory to save output files and plots.
    group : str, optional
        The group name for extracting factors and metrics (default: "group_1").
    n_clusters_list : list of int or int, optional
        List of cluster numbers to sweep for clustering evaluation (default: None, uses range 2-19).
    obs_keys : list of str
        Observation keys (e.g. ['major_brain_region', 'cell_type']) to be used for comparisons and metrics.
        This parameter is required (no default); if label_key is None the first entry is used for classification labels.
    label_key : str, optional
        Observation key used for classification labels. If None, defaults to the first element of obs_keys.
    """
    out_dir = _ensure_dir(output_dir)

    # create central plots and stats directories
    plots_dir = _ensure_dir(out_dir / "plots")
    stats_dir = _ensure_dir(out_dir / "stats")

    if obs_keys is None or not isinstance(obs_keys, (list, tuple)) or len(obs_keys) == 0:
        raise ValueError("You must provide obs_keys (list of observation keys).")

    if label_key is None:
        label_key = obs_keys[0]

    # 1) MOFA model evaluation (plots + R2) -> save plots into plots_dir
    mofa_results = evaluate_single_mofa_model(model, plots_dir, group=group)

    # 2) weights distribution plot -> save into plots_dir
    weights_path_result = plot_weights_distribution(model, plots_dir)
    if isinstance(weights_path_result, dict) and "error" in weights_path_result:
        weights_path = ""
    else:
        weights_path = weights_path_result

    # 3) prepare factors aligned to lrdata (filtered)
    factors = model.get_factors()
    if isinstance(factors, dict):
        factors = factors[group]
    # align to non-na observations in lrdata
    lrdata_clean = lrdata.copy()
    # filter by all user-specified keys if present
    for key in obs_keys:
        if key in lrdata_clean.obs:
            lrdata_clean = lrdata_clean[~lrdata_clean.obs[key].isna()].copy()

    common_idx = factors.index.intersection(lrdata_clean.obs_names)
    factors_sub = factors.loc[common_idx].copy()
    lrdata_sub = lrdata_clean[common_idx, :].copy()

    # 4) classification using user-provided label_key -> save CSVs into stats_dir
    try:
        class_metrics, class_report_path = classification_on_factors(factors_sub, lrdata_sub, stats_dir, label_key=label_key)
    except Exception as e:
        class_metrics = {"error": str(e)}
        class_report_path = ""

    # 5) allow user configuration via lrdata.uns
    cfg = {}
    if hasattr(lrdata, "uns") and isinstance(lrdata.uns, dict):
        cfg = lrdata.uns.get("evaluation", {}) or {}

    spatial_key = cfg.get("spatial_key", globals().get("spatial_key", "X_spatial_coords"))

    # default cluster number is 6 if not provided; allow n_clusters_list int to override
    chosen_n_clusters = cfg.get("n_clusters", globals().get("n_clusters", None))
    if chosen_n_clusters is None:
        if isinstance(n_clusters_list, int):
            chosen_n_clusters = int(n_clusters_list)
        else:
            chosen_n_clusters = 6
    else:
        chosen_n_clusters = int(chosen_n_clusters)

    # ensure the spatial coordinates are available under the expected key used by downstream functions
    if spatial_key != "X_spatial_coords":
        if spatial_key in getattr(lrdata, "obsm", {}):
            # copy the provided spatial coordinates into the expected key so downstream functions use them
            lrdata.obsm["X_spatial_coords"] = lrdata.obsm[spatial_key].copy()
        else:
            # fallback to existing 'X_spatial_coords' if present; otherwise leave as-is
            if "X_spatial_coords" in getattr(lrdata, "obsm", {}):
                lrdata.obsm["X_spatial_coords"] = lrdata.obsm["X_spatial_coords"].copy()

    # enforce/create a 'cluster' obs column using the chosen fixed number of clusters (overwrites existing cluster column)
    try:
        # use the already aligned factors_sub and lrdata_sub
        drop_cols = [c for c in ["cluster", "anno"] if c in factors_sub.columns]
        X_cluster = factors_sub.drop(columns=drop_cols).values
        X_cluster_std = StandardScaler().fit_transform(X_cluster)
        km_fixed = KMeans(n_clusters=chosen_n_clusters, random_state=42)
        fixed_labels = km_fixed.fit_predict(X_cluster_std).astype(str)

        # assign to lrdata_sub.obs and propagate back to the main lrdata.obs for the matching indices
        lrdata_sub.obs["cluster"] = fixed_labels
        # ensure lrdata.obs has the 'cluster' column and update rows corresponding to lrdata_sub
        valid_indices = [idx for idx in lrdata_sub.obs_names if idx in lrdata.obs.index]
        lrdata.obs.loc[valid_indices, "cluster"] = pd.Series(fixed_labels, index=lrdata_sub.obs_names).reindex(valid_indices).values
    except Exception:
        pass

    # 6) clustering sweep (pass user-provided obs keys) -> save CSV into stats_dir
    clustering_df, clustering_csv = clustering_sweep(factors_sub, lrdata_sub, stats_dir,
                                                     n_clusters_list=n_clusters_list,
                                                     obs_keys=obs_keys)

    # 7) summary DataFrame (one-row)
    # Find the index (number of clusters) with the highest silhouette score for spatial clustering
    silhouette_scores = clustering_df["silhouette_spatial"]
    has_valid_scores = silhouette_scores.notna().any()
    if has_valid_scores:
        best_n = int(silhouette_scores.idxmax())
    else:
        best_n = np.nan

    summary = {
        "mean_r2": mofa_results.get("mean_r2", np.nan),
        "classification_accuracy": class_metrics.get("accuracy", np.nan) if isinstance(class_metrics, dict) else np.nan,
        "classification_auc_macro": class_metrics.get("auc_macro", np.nan) if isinstance(class_metrics, dict) else np.nan,
        "classification_auc_weighted": class_metrics.get("auc_weighted", np.nan) if isinstance(class_metrics, dict) else np.nan,
        "clustering_csv": clustering_csv,
        "used_obs_keys": ",".join(map(str, obs_keys)),
        "used_label_key_for_classification": label_key,
    }
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(stats_dir / "summary_metrics.csv", index=False)
    results = {
        "summary": summary_df,
        "clustering": clustering_df,
        "classification_metrics": class_metrics,
        "classification_report_path": class_report_path,
        "plots_dir": str(plots_dir),
        "stats_dir": str(stats_dir),
    }

    return results


In [6]:
import os.path
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
import mofaflex as mfl
from plotnine import *
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from umap import UMAP
from sklearn.manifold import TSNE

In [57]:
models_path = Path("/g/stegle/aalsayah/repos/data/merfish/mouse_brain/mofa_models")
model= mfl.MOFAFLEX.load(models_path / "WB_MERFISH_animal2_coronal_Normal.hdf5")
merfish_path = Path("/g/stegle/aalsayah/repos/data/merfish/mouse_brain/lrdata")
lrdata = sc.read(merfish_path / "WB_MERFISH_animal2_coronal_lrdata.h5ad")

In [62]:
results_normal = evaluate_pipeline(model, lrdata, '/g/stegle/aalsayah/repos/liana-py/docs/source/unfinished_temp_notebooks/MOFA_Flex/Normal_report',
                            group="group_1",
                      n_clusters_list=None,
                      obs_keys=["major_brain_region", "cell_type"],
                      label_key=None)

/g/stegle/aalsayah/Miniforge3/envs/mofa_flex/lib/python3.11/site-packages/plotnine/ggplot.py:606: PlotnineWarning: Saving 8 x 8 in image.
/g/stegle/aalsayah/Miniforge3/envs/mofa_flex/lib/python3.11/site-packages/plotnine/ggplot.py:607: PlotnineWarning: Filename: /g/stegle/aalsayah/repos/liana-py/docs/source/unfinished_temp_notebooks/MOFA_Flex/Normal_report/plots/factor_correlation.png
/g/stegle/aalsayah/Miniforge3/envs/mofa_flex/lib/python3.11/site-packages/plotnine/ggplot.py:606: PlotnineWarning: Saving 8 x 8 in image.
/g/stegle/aalsayah/Miniforge3/envs/mofa_flex/lib/python3.11/site-packages/plotnine/ggplot.py:607: PlotnineWarning: Filename: /g/stegle/aalsayah/repos/liana-py/docs/source/unfinished_temp_notebooks/MOFA_Flex/Normal_report/plots/variance_explained.png


In [60]:
results_normal

{'summary':     mean_r2  classification_accuracy  classification_auc_macro  \
 0  0.713457                 0.340057                  0.745723   
 
    classification_auc_weighted  \
 0                     0.737527   
 
                                       clustering_csv  \
 0  /g/stegle/aalsayah/repos/liana-py/docs/source/...   
 
                   used_obs_keys used_label_key_for_classification  
 0  major_brain_region,cell_type                major_brain_region  ,
 'clustering':             silhouette_spatial  silhouette_major_brain_region  \
 n_clusters                                                      
 2                    -0.055928                       0.008107   
 3                    -0.060429                       0.008107   
 4                    -0.095051                       0.008107   
 5                    -0.110390                       0.008107   
 6                    -0.118792                       0.008107   
 7                    -0.136949                   

In [ ]:
results_nonneg = evaluate_pipeline(model, lrdata, '/g/stegle/aalsayah/repos/liana-py/docs/source/unfinished_temp_notebooks/MOFA_Flex/nonneg_report',
                            group="group_1",
                      n_clusters_list=None,
                      obs_keys=["major_brain_region", "cell_type"],
                      label_key=None)

In [ ]:
results_GP_SnS = evaluate_pipeline(model, lrdata, '/g/stegle/aalsayah/repos/liana-py/docs/source/unfinished_temp_notebooks/MOFA_Flex/GP_SnS_report',
                            group="group_1",
                      n_clusters_list=None,
                      obs_keys=["major_brain_region", "cell_type"],
                      label_key=None)

# Merge

In [43]:
results = {
    "nonneg": results_nonneg,
    "GP_SnS": results_GP_SnS,
    "Normal": results_normal
}

### Summary

In [44]:
dfs = []
for name, obj in results.items():
    df = pd.DataFrame(obj["summary"]).copy()
    df["source"] = name
    df = df[["source"] + [c for c in df.columns if c != "source"]]
    dfs.append(df)

df_summary = pd.concat(dfs, ignore_index=True)


In [48]:
df_summary

,source,mean_r2,classification_accuracy,classification_auc_macro,classification_auc_weighted,clustering_csv,join_counts_csv,used_obs_keys,used_label_key_for_classification
0,nonneg,0.707430,0.308239,0.734753,0.729942,/g/stegle/aalsayah/repos/liana-py/docs/source/...,/g/stegle/aalsayah/repos/liana-py/docs/source/...,"major_brain_region,cell_type",major_brain_region
1,GP_SnS,0.727423,0.417330,0.785158,0.797266,/g/stegle/aalsayah/repos/liana-py/docs/source/...,/g/stegle/aalsayah/repos/liana-py/docs/source/...,"major_brain_region,cell_type",major_brain_region
2,Normal,0.713457,0.340057,0.745723,0.737527,/g/stegle/aalsayah/repos/liana-py/docs/source/...,/g/stegle/aalsayah/repos/liana-py/docs/source/...,"major_brain_region,cell_type",major_brain_region


### Clustering

In [45]:
dfs = []
for name, obj in results.items():
    df = pd.DataFrame(obj["clustering"]).copy()
    df["source"] = name
    df = df[["source"] + [c for c in df.columns if c != "source"]]
    dfs.append(df)

df_clustering = pd.concat(dfs)
df_clustering.reset_index(inplace=True)  # keep n_clusters as a column
df_clustering = df_clustering.rename(columns={"index": "n_clusters"})


In [ ]:
df_clustering[df_clustering["source"] == "GP_SnS"] #NMI - ARI - spatial coherence - AWS -

,n_clusters,source,silhouette_spatial,silhouette_major_brain_region,nmi_cluster_major_brain_region,ami_cluster_major_brain_region,hom_cluster_major_brain_region,ari_cluster_major_brain_region,spatial_coherence_score_major_brain_region,silhouette_cell_type,nmi_cluster_cell_type,ami_cluster_cell_type,hom_cluster_cell_type,ari_cluster_cell_type,spatial_coherence_score_cell_type,nmi_major_brain_region_vs_cell_type,ami_major_brain_region_vs_cell_type,hom_major_brain_region_vs_cell_type,ari_major_brain_region_vs_cell_type
18,2,GP_SnS,0.108486,0.008107,0.084582,0.084292,0.058956,0.087978,0.970792,-0.531486,0.008353,0.007427,0.006490,0.010088,0.599557,0.136866,0.133367,0.117116,0.183031
19,3,GP_SnS,0.055899,0.008107,0.119549,0.119060,0.095234,0.127566,0.970792,-0.531486,0.055666,0.054229,0.051043,0.063596,0.599557,0.136866,0.133367,0.117116,0.183031
20,4,GP_SnS,-0.035507,0.008107,0.133373,0.132714,0.116707,0.090441,0.970792,-0.531486,0.058981,0.057106,0.060585,0.032085,0.599557,0.136866,0.133367,0.117116,0.183031
21,5,GP_SnS,-0.037420,0.008107,0.125092,0.124244,0.114941,0.159271,0.970792,-0.531486,0.052367,0.050074,0.057015,0.052643,0.599557,0.136866,0.133367,0.117116,0.183031
22,6,GP_SnS,-0.054925,0.008107,0.087573,0.086509,0.083960,0.126391,0.970792,-0.531486,0.059920,0.057270,0.068598,0.071548,0.599557,0.136866,0.133367,0.117116,0.183031
23,7,GP_SnS,-0.079599,0.008107,0.106434,0.105262,0.108917,0.095641,0.970792,-0.531486,0.061587,0.058657,0.076098,0.047153,0.599557,0.136866,0.133367,0.117116,0.183031
24,8,GP_SnS,-0.126616,0.008107,0.126684,0.125303,0.126254,0.116747,0.970792,-0.531486,0.063683,0.060307,0.076297,0.075686,0.599557,0.136866,0.133367,0.117116,0.183031
25,9,GP_SnS,-0.126818,0.008107,0.136408,0.134941,0.144546,0.148264,0.970792,-0.531486,0.060807,0.057196,0.078241,0.066486,0.599557,0.136866,0.133367,0.117116,0.183031
26,10,GP_SnS,-0.170866,0.008107,0.161859,0.160288,0.175099,0.170607,0.970792,-0.531486,0.079044,0.075205,0.104166,0.078590,0.599557,0.136866,0.133367,0.117116,0.183031
27,11,GP_SnS,-0.172376,0.008107,0.129861,0.128111,0.145544,0.102715,0.970792,-0.531486,0.061012,0.056862,0.083745,0.046278,0.599557,0.136866,0.133367,0.117116,0.183031


### Join Counts

In [46]:
dfs = []
for name, obj in results.items():
    df = pd.DataFrame(obj["join_counts"]).copy()
    df["source"] = name
    df = df[["source"] + [c for c in df.columns if c != "source"]]
    dfs.append(df)

df_join_counts = pd.concat(dfs, ignore_index=True)


In [55]:
df_join_counts[df_join_counts["source"] == "nonneg"]

,source,obs_key,region,BB,WW,BW,expected_BB,p_value
0,nonneg,cluster,0,898.0,11619.0,2896.0,333.811812,0.001
1,nonneg,cluster,1,221.0,12602.0,2590.0,158.820821,0.001
2,nonneg,cluster,2,4874.0,4619.0,5920.0,4452.021021,0.001
3,nonneg,cluster,3,213.0,13103.0,2097.0,100.755756,0.001
4,nonneg,cluster,4,224.0,13636.0,1553.0,72.320320,0.001
5,nonneg,cluster,5,591.0,13094.0,1728.0,63.036036,0.001
6,nonneg,major_brain_region,Cortical_subplate,4743.0,150191.0,475.0,197.639640,0.001
7,nonneg,major_brain_region,Fiber_tracts,14252.0,138556.0,2601.0,1669.721722,0.001
8,nonneg,major_brain_region,Hippocampus,3726.0,151273.0,410.0,82.597598,0.001
9,nonneg,major_brain_region,Hypothalamus,16725.0,137886.0,798.0,1702.298298,0.001


### Classification Report

In [47]:
dfs = []
for name, obj in results.items():
    report_dict = obj["classification_metrics"]["classification_report"]
    df = pd.DataFrame(report_dict).T  # transpose
    df["source"] = name
    df = df[["source"] + [c for c in df.columns if c != "source"]]
    dfs.append(df)

df_classification_report = pd.concat(dfs, ignore_index=True)


----

# Load models

In [3]:
models_path = Path("/g/stegle/aalsayah/repos/data/merfish/mouse_brain/mofa_models")

In [4]:
model= mfl.MOFAFLEX.load(models_path / "WB_MERFISH_animal2_coronal_Normal.hdf5")


# Overview analysis

In [ ]:
from plotnine.ggplot import ggplot  # to check object type

def evaluate_single_mofa_model(model, output_dir=None, group="group_1", show_plots=True):
    """
    Evaluate a single trained MOFAFLEX model.
    Handles both plotnine (ggplot) and matplotlib plots.
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    results = {}

    # --- Factor correlation ---
    print("Plotting factor correlation...")
    fig1 = mfl.pl.factor_correlation(model)
    if output_dir:
        fc_path = output_dir / "factor_correlation.png"
        if isinstance(fig1, ggplot):
            fig1.save(fc_path, dpi=300)
        else:
            fig1.savefig(fc_path, bbox_inches="tight")
        results["factor_correlation_path"] = str(fc_path)
    if show_plots:
        print(fig1)
    else:
        plt.close('all')

    # # --- quality control for SnS priors ---
    # print("Plotting weight_sparsity_histogram (QC for SnS priors)...")
    # fig1 = mfl.pl.weight_sparsity_histogram(model)
    # if output_dir:
    #     fc_path = output_dir / "weight_sparsity_histogram.png"
    #     if isinstance(fig1, ggplot):
    #         fig1.save(fc_path, dpi=300)
    #     else:
    #         fig1.savefig(fc_path, bbox_inches="tight")
    #     results["weight_sparsity_histogram_path"] = str(fc_path)
    # if show_plots:
    #     print(fig1)
    # else:
    #     plt.close('all')

    # --- Variance explained ---
    print("Plotting variance explained...")
    fig2 = mfl.pl.variance_explained(model, figsize=(8, 8))
    if output_dir:
        ve_path = output_dir / "variance_explained.png"
        if isinstance(fig2, ggplot):
            fig2.save(ve_path, dpi=300)
        else:
            fig2.savefig(ve_path, bbox_inches="tight")
        results["variance_explained_path"] = str(ve_path)
    if show_plots:
        print(fig2)
    else:
        plt.close('all')

    # --- Compute R² ---
    print("Computing R²...")
    r2 = model.get_r2(total=True)
    mean_r2 = float(r2[group].mean())
    results["mean_r2"] = mean_r2
    print(f"✅ Mean R² for {group}: {mean_r2:.4f}")

    # --- Save metrics file ---
    if output_dir:
        metrics_path = output_dir / "metrics.txt"
        with open(metrics_path, "w") as f:
            f.write(f"Mean R² ({group}): {mean_r2:.4f}\n")
        results["metrics_path"] = str(metrics_path)

    return results


In [ ]:
results = evaluate_single_mofa_model(model, output_dir="/g/stegle/aalsayah/repos/data/merfish/mouse_brain/mofa_models/evaluation_results2", group="group_1", show_plots=True)

## 3.Weights dist

In [ ]:
weights_dict= model.get_weights()
plt.figure(figsize=(12, 6))

# Loop through each view/key
for key, df in weights_dict.items():
    # Flatten all weights from that DataFrame into 1D array
    values = df.values.flatten()
    sns.kdeplot(values, label=key, fill=False)  # distribution curve

plt.title("Distribution of Weights per View")
plt.xlabel("Weight Value")
plt.ylabel("Density")
plt.legend(title="View/Key", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Cluster factors

In [7]:
merfish_path = Path("/g/stegle/aalsayah/repos/data/merfish/mouse_brain/lrdata")
lrdata = sc.read(merfish_path / "WB_MERFISH_animal2_coronal_lrdata.h5ad")

In [ ]:
from sklearn.metrics import *
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

In [ ]:
factors = model.get_factors()
factors = factors["group_1"]

In [ ]:
lrdata_clean = lrdata[~lrdata.obs["cell_type"].isna()].copy()
lrdata_clean = lrdata_clean[~lrdata_clean.obs["major_brain_region"].isna()].copy()

In [ ]:
def spatial_coherence_score(X, labels, k=5):
    X = np.asarray(X)
    labels = np.asarray(labels)

    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
    _, indices = nbrs.kneighbors(X)

    # exclude each point itself (indices[:,0])
    indices = indices[:, 1:]
    
    same_cluster_counts = np.sum(labels[indices] == labels[:, None], axis=1)
    scs = np.mean(same_cluster_counts / k)
    return scs


In [ ]:
# choose cluster counts to evaluate
n_clusters_list = list(range(2, 20))  # change as needed

# align factors to the filtered AnnData (lrdata_clean2)
common_idx = factors.index.intersection(lrdata_clean.obs_names)
factors_sub = factors.loc[common_idx].copy()
lrdata_sub = lrdata_clean[common_idx, :].copy()

# remove columns that are not factor values
drop_cols = [c for c in ["cluster", "anno"] if c in factors_sub.columns]
X_f = factors_sub.drop(columns=drop_cols)

# standardize
X_std = StandardScaler().fit_transform(X_f.values)

# spatial coords aligned to same ordering
X_spatial = lrdata_sub.obsm["X_spatial_coords"]
# ensure same ordering: lrdata_clean.obs_names should equal common_idx order
if not (list(common_idx) == list(lrdata_sub.obs_names)):
    # reorder spatial coords to match factors_sub
    X_spatial = lrdata_sub.obsm["X_spatial_coords"][ [lrdata_sub.obs_names.get_loc(i) for i in common_idx], : ]

results = []
for k in tqdm(n_clusters_list, desc="kmeans sweep"):
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_std)

    # compute metrics (use lrdata_clean labels where appropriate)
    # silhouette on spatial coords using cluster labels
    try:
        sil_spatial = float(silhouette_score(X_spatial, labels))
        sil_major_brain_region = float(silhouette_score(X_spatial, lrdata_sub.obs["major_brain_region"].astype(str).values))
    except Exception:
        sil_spatial = float("nan")

    # NMI
    nmi_celltype = float(normalized_mutual_info_score(lrdata_sub.obs["cell_type"].astype(str).values, labels))
    nmi_region = float(normalized_mutual_info_score(lrdata_sub.obs["major_brain_region"].astype(str).values, labels))
    nmi_ground_truth = float(normalized_mutual_info_score(lrdata_sub.obs["major_brain_region"].astype(str).values, lrdata_sub.obs["cell_type"].astype(str).values))

    # AMI
    ami_celltype = float(adjusted_mutual_info_score(lrdata_sub.obs["cell_type"].astype(str).values, labels))
    ami_region = float(adjusted_mutual_info_score(lrdata_sub.obs["major_brain_region"].astype(str).values, labels))
    ami_ground_truth = float(adjusted_mutual_info_score(lrdata_sub.obs["major_brain_region"].astype(str).values, lrdata_sub.obs["cell_type"].astype(str).values))

    # Homogeneity (true, pred)
    hom_celltype = float(homogeneity_score(lrdata_sub.obs["cell_type"].astype(str).values, labels))
    hom_region = float(homogeneity_score(lrdata_sub.obs["major_brain_region"].astype(str).values, labels))
    hom_ground_truth = float(homogeneity_score(lrdata_sub.obs["major_brain_region"].astype(str).values, lrdata_sub.obs["cell_type"].astype(str).values))

    # ARI
    ari_celltype = float(adjusted_rand_score(lrdata_sub.obs["cell_type"].astype(str).values, labels))
    ari_region = float(adjusted_rand_score(lrdata_sub.obs["major_brain_region"].astype(str).values, labels))
    ari_ground_truth = float(adjusted_rand_score(lrdata_sub.obs["major_brain_region"].astype(str).values, lrdata_sub.obs["cell_type"].astype(str).values))


    # Spatial coherence score (uses the notebook-defined function)
    try:
        scs = float(spatial_coherence_score(X_spatial, labels, k=3))
        scs_major_brain_region = float(spatial_coherence_score(X_spatial, lrdata_sub.obs["major_brain_region"].astype(str).values, k=3))
    except Exception:
        scs = float("nan")

    results.append({
        "n_clusters": k,
        "silhouette_spatial": sil_spatial,
        "silhouette_major_brain_region": sil_major_brain_region,
        "nmi_cluster_cell_type": nmi_celltype,
        "nmi_cluster_major_brain_region": nmi_region,
        "nmi_major_brain_region_vs_cell_type": nmi_ground_truth,
        "ami_cluster_cell_type": ami_celltype,
        "ami_cluster_major_brain_region": ami_region,
        "ami_major_brain_region_vs_cell_type": ami_ground_truth,
        "hom_cluster_cell_type": hom_celltype,
        "hom_cluster_major_brain_region": hom_region,
        "hom_major_brain_region_vs_cell_type": hom_ground_truth,
        "ari_cluster_cell_type": ari_celltype,
        "ari_cluster_major_brain_region": ari_region,
        "ari_major_brain_region_vs_cell_type": ari_ground_truth,
        "spatial_coherence_score": scs,
        "spatial_coherence_score_major_brain_region": scs_major_brain_region
    })

results_df = pd.DataFrame(results).set_index("n_clusters")
results_df = results_df.sort_index()

# show results and optionally save
print(results_df)
#results_df.to_csv("clustering_metrics_sweep.csv")

# Regression Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
obskey1 = lrdata.obs["major_brain_region"]
factors['anno'] = obskey1.reindex(factors.index)
factors = factors.dropna()
print(factors['anno'].isna().sum())
len(factors)

In [ ]:
X = factors.drop(columns=['cluster', 'anno']).values
X

In [ ]:
anno = factors["anno"].values
anno

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
   X, anno, test_size=0.3, random_state=42, stratify=anno
)

In [ ]:
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=200,
    class_weight="balanced"
)

In [ ]:
# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

#### 1. If threr are 2 classes only (healthy vs. disease)

In [ ]:
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Calculate AUC
auc = roc_auc_score(y_test, y_pred_proba)
print("AUC:", auc)


fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba, pos_label="222_REF")
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

#### 2. If classes are more than 2:

In [ ]:
# Get class probabilities 
y_proba = clf.predict_proba(X_test)  

# Compute macro-average AUC (each class equally weighted)
auc_macro = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
print(f"Multiclass ROC AUC (OvR, macro): {auc_macro:.4f}")

# weights by class support:
auc_weighted = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
print(f"Multiclass ROC AUC (OvR, weighted): {auc_weighted:.4f}")


# spatial autocorrelation using join-count statistics

Run 3 times, one for obs key 1, obs key2 and cluster

In [ ]:
import geopandas as gpd
import libpysal as lps
from esda.moran import Moran_Local
import numpy as np
from esda.join_counts import Join_Counts
from scipy.sparse import csr_matrix

In [ ]:
coords = lrdata.obsm["X_spatial_coords"]
regions = lrdata.obs["cluster"]
valid = ~pd.isna(regions)
coords_clean = coords[valid]
regions_clean = np.array(regions)[valid]

In [ ]:
from libpysal.weights import DistanceBand
import esda

# build distance neighbors (choose threshold based on tissue scale)
w = DistanceBand(coords_clean, threshold=30, binary=True)  # ~30 microns example

unique_regions = np.unique(regions_clean)

In [ ]:
results = []

for r in unique_regions:
    y = (regions_clean == r).astype(int)  # one-vs-rest binary mask
    jc = esda.join_counts.Join_Counts(y, w)
    
    results.append({
        "region": r,
        "BB": jc.bb,            # region with region
        "WW": jc.ww,            # other with other
        "BW": jc.bw,            # mixing
        "expected_BB": jc.mean_bb,
        "p_value": jc.p_sim_bb
    })

# convert to DataFrame for display
import pandas as pd
df = pd.DataFrame(results)
print(df.sort_values("p_value"))